# 第4日 ノートブック02：YOLO モデルを学習する

# Day 4 Notebook 02: Training a YOLO Model

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. 学習の流れ（epoch・batch・loss）を説明できる | 1. Explain the training loop (epoch, batch, loss) |
| 2. ファインチューニングの意味を説明できる | 2. Explain what fine-tuning is |
| 3. CPU で現実的な時間で学習を実行できる | 3. Run a CPU-feasible training job |
| 4. 学習の進み具合を読み取れる | 4. Read training progress |
| 5. 学習結果（重み・設定・指標・ログ）を保存できる | 5. Save weights, config, metrics and logs |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 115 分 / 115 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第4日ノートブック01 を終え、`assets/yolo_dataset` が生成されていること。 / You finished Day 4 Notebook 01 and `assets/yolo_dataset` exists. |
| 必要なライブラリ / Required libraries | torch, ultralytics, opencv-python, numpy, matplotlib, pandas |
| 使用カーネル / Jupyter kernel | aivision |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第4日 / Day 4](../../COURSE_SYLLABUS.md#day4) |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| 学習済み重みファイル | The trained weight file |
| 学習ログと指標 | Training logs and metrics |
| 学習設定の記録 | A record of the training configuration |

## このノートブックの位置 / Where this notebook sits

| 項目 / Item | 内容 / Details |
| --- | --- |
| 日 / Day | 第4日 / Day 4 |
| この日の順番 / Position today | 2 / 3（この日のノートブックのうち）/ of today's notebooks |
| 所要時間 / Duration | 115 分 / 115 minutes |
| 前提 / Prerequisites | `Day04_01_YOLO_Dataset_Annotation.ipynb`、`assets/yolo_dataset` 生成済み |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第4日 / Day 4](../../COURSE_SYLLABUS.md#day4) |

1 日の時間割は `COURSE_SYLLABUS.md` に 1 つだけあります。このノートブックはその一部で、
自分の時間割を持ちません。時刻が分からなくなったら、まずそこを見てください。

The day's timetable exists once, in `COURSE_SYLLABUS.md`. This notebook is one part of it and
does not keep its own schedule; look there first when you are unsure of the time.


### 予想して、実行して、説明する / Predict, run, inspect, explain

epoch はデータを1周、batch は一度に渡す画像群です。更新回数とは区別します。 重要なセルの前に、入力・操作・出力を一文ずつ説明してください。
予想を書いて実行し、値または図を観察します。batch だけ（予測してから実行）を変え、もう一度予想して実行します。
意図的なエラーと未完成の演習は、その指示に従って取り組みます。

An epoch is a pass through data; a batch is a group processed together. Distinguish optimizer updates. Before an important cell, describe its input, operation and output in one sentence each.
Predict, run, inspect the values/image and explain why. Change only batch (predict before running), predict and rerun.
Follow the instructions for deliberately erroneous cells and unsolved exercises.


---

## Part 0　準備 / Preparation

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd().resolve()
while not ((COURSE_ROOT / "assets").is_dir() and (COURSE_ROOT / "notebooks").is_dir()):
    if COURSE_ROOT.parent == COURSE_ROOT:
        raise FileNotFoundError(
            "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
            "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
        )
    COURSE_ROOT = COURSE_ROOT.parent

ASSETS = COURSE_ROOT / "assets"
sys.path.insert(0, str(ASSETS / "helpers"))
import aivision

print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("cv2", "opencv-python"),
    ("torch", "torch"),
    ("ultralytics", "ultralytics"),
    ("pandas", "pandas"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
FONT_FAMILY = aivision.configure_japanese_font()
aivision.set_plot_style()

if FONT_FAMILY is None:
    print("日本語フォントが見つかりません。図のタイトルは英字になります。")
    print("No Japanese font found; figure titles fall back to English.")
else:
    print("図の日本語フォント / figure font:", FONT_FAMILY)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

def show(images, titles=None, figsize=(14, 5)):
    """画像を並べて表示する / display images side by side."""
    if not isinstance(images, (list, tuple)):
        images = [images]
    if titles is None:
        titles = [""] * len(images)
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image)
        ax.set_title(title, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()


def detections_to_frame(result):
    """検出結果を pandas の表にする / convert detections into a DataFrame.

    ``result.boxes.data`` の 6 列は
    (x1, y1, x2, y2, confidence, class_id) です。
    """
    columns = ["class_id", "name", "confidence", "x1", "y1", "x2", "y2"]
    boxes = result.boxes
    if len(boxes) == 0:
        return pd.DataFrame(columns=columns)

    data = boxes.data.cpu().numpy()
    return pd.DataFrame({
        "class_id": data[:, 5].astype(int),
        "name": [result.names[int(class_id)] for class_id in data[:, 5]],
        "confidence": data[:, 4].round(4),
        "x1": data[:, 0].round(2),
        "y1": data[:, 1].round(2),
        "x2": data[:, 2].round(2),
        "y2": data[:, 3].round(2),
    })

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import shutil
from pathlib import Path

MODELS_DIR = ASSETS / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

def load_pretrained_yolo(model_name="yolo26n.pt"):
    """学習済みモデルを読み込む。初回だけダウンロードする。
    Load a pretrained model, downloading it only on the first run."""
    from ultralytics import YOLO

    weights_path = MODELS_DIR / model_name
    if not weights_path.is_file():
        print(f"初回のみダウンロードします / downloading {model_name} (first run only)...")
        print("  ネットワークに接続できない場合は、次を確認してください:")
        print("  - 社内プロキシの設定 / corporate proxy settings")
        print("  - もう一度このセルを実行する / run this cell again")
        model = YOLO(model_name)          # ダウンロードはカレントフォルダに行われる
        downloaded = Path(model_name)
        if downloaded.is_file():
            shutil.move(str(downloaded), str(weights_path))
            print("保存しました / cached to:", weights_path)
        else:
            raise RuntimeError(
                f"モデルを取得できませんでした / could not obtain {model_name}. "
                "ネットワークを確認して再実行してください / check the network and retry."
            )
    return YOLO(str(weights_path))

---

## Part 1　学習とは何か / What training is

### epoch と batch / epochs and batches

![](../../assets/diagrams/teach_epoch_batch.png)

図：データローダが 8 枚ずつの batch を渡し、8 batch 分の勾配をためて重みを 1 回更新する流れ
（この研修の設定 `batch=8`・`nbs=64`）。

Figure: the data loader passes batches of 8 images, and the gradients of 8 batches are accumulated
into one weight update (this course's `batch=8` and `nbs=64`).

**気づいてほしいこと**

- **batch（バッチ）** … データローダが一度にまとめて読み込む枚数。この研修では 8 枚。勾配はここで計算する
- **optimizer step（重みの更新）** … ためた勾配で重みを 1 回動かすこと。この設定では 8 batch ごとに 1 回
- **勾配の累積（gradient accumulation）** … 何バッチ分の勾配をためてから 1 回更新するか（`nbs` ÷ `batch`）
- **epoch** … 全学習画像を 1 周すること。この研修では 15 周

64 枚 ÷ 8 枚 = 8 batch が 1 epoch です。勾配を 8 batch 分ためてから更新するので、
**1 batch = 1 回の更新ではありません**。さらに学習の最初には **warmup**（学習率とためる数を
徐々に目標へ近づける期間）があり、その間はためる数が 1 から段階的に増えます。
だから「64 ÷ 8 = 8 回更新」と数えることはできません。更新回数は学習器が決めます。

loss は「今どれだけ間違えているか」で、下がるほど学習が進んでいます。

**What to notice**

- **batch**: how many images the data loader delivers at once — 8 here, and the gradient is computed per batch.
- **optimizer step**: one move of the weights with the accumulated gradients — every 8 batches here.
- **gradient accumulation**: how many batches of gradients are summed before that update (`nbs` / `batch`).
- **epoch**: one pass over the whole training set — 15 here.

64 images / 8 per batch = 8 data-loader batches per epoch. Because 8 batches of gradients are
accumulated, **one batch is not one update**. A **warmup** period also ramps the learning rate and
the accumulated batch count from 1 up to the target, so "64 / 8 = 8 updates" is not a valid count;
the trainer decides it.

The loss says how wrong the model is right now, and it falls as learning progresses.

**考えてみてください / Try to answer**

- 学習画像が 64 枚、batch が 8 枚のとき、1 epoch のデータローダの反復は何回ですか。そのうち
  何回が重みの更新になるかは、何で決まりますか。
- With 64 training images and a batch of 8, how many data-loader iterations are in one epoch, and
  what decides how many of them become weight updates?


#### 学習の繰り返し
学習は、次の 4 ステップの **繰り返し** です。

1. **予測する** — 今のモデルで、学習画像を推論する
2. **間違いを測る** — 正解との差を数値にする。これを **損失（loss）** と呼ぶ
3. **少し直す** — 損失が小さくなる方向に、モデルの数値を少し動かす
4. **繰り返す** — 1 周分（全学習画像）を終えると 1 **エポック**

ステップ 3 の「直す」は、毎回起きるとは限りません。勾配を何バッチ分ためてから 1 回更新するかは
設定で決まります（この研修の既定は `nbs=64` と `batch=8` で 8 バッチ分）。

損失（loss）が下がっていれば、学習は進んでいます。
ただし **学習用の損失が下がり続けても、実力が上がるとは限りません**（過学習）。

#### The training loop
Training is a **loop** of four steps:

1. **predict** — run the current model on the training images
2. **measure the error** — compare with the truth; the number is the **loss**
3. **adjust a little** — nudge the model's numbers in the direction that reduces the loss
4. **repeat** — one full pass over the training set is one **epoch**

Step 3 does not happen on every iteration. How many batches of gradients are summed before one
update is a setting (this course's defaults are `nbs=64` and `batch=8`, so 8 batches).

If the loss falls, learning is progressing. But a falling *training* loss does not guarantee better
real-world skill — that is overfitting.


### 1 つの batch で起きること / what one batch does

![](../../assets/diagrams/teach_training_update.png)

図：予測 → 損失 → 勾配 → 更新 の 1 サイクルと、batch・step・epoch の数え方（重みの変化は例示用）。

Figure: one cycle of forward → loss → backward → update, and how batch, step and epoch are counted (weight changes illustrative).

**気づいてほしいこと**

バッチ（データをまとめて読む単位）と、重みを更新する回数は同じとは限りません。この研修の設定は `nbs=64`・`batch=8` で、8 バッチ分の勾配をためてから 1 回更新します（gradient accumulation）。64 枚を 1 周すると、データローダは 8 回まわり、重みの更新は 1 回です。学習の最初は warmup で更新の幅が変わります。

**What to notice**

A batch (how many samples the loader reads together) is not necessarily how often the weights update. This course uses `nbs=64` and `batch=8`, accumulating the gradients of 8 batches into one update: one pass over 64 images gives 8 loader iterations and 1 weight update. Warmup changes the step size at the start.

**考えてみてください / Try to answer**

- batch=8 で 64 枚を 1 周すると、重みは何回更新されますか（nbs=64 のとき）。
- With batch=8 and 64 images per epoch, how many weight updates happen when nbs=64?


#### 用語を一度に整理する

| 用語 | 意味 | この研修での値 |
|---|---|---|
| エポック（epoch） | 全学習画像を 1 周すること | 15 |
| バッチサイズ（batch） | データローダが一度に読み込む枚数 | 8 |
| 勾配の累積（gradient accumulation） | 何バッチ分の勾配をためてから 1 回更新するか | `nbs=64` と `batch=8` から 8 バッチ |
| warmup | 学習の最初に、学習率とためる数を目標へ近づけていく期間 | 3 epoch 分 |
| 学習率（learning rate） | 1 回の修正の大きさ | 0.01 から徐々に下げる |
| 損失（loss） | 正解との差。小さいほど良い | 学習で下がる |
| 学習（train） | 数値を決める作業 | — |
| 検証（val） | 学習中に実力を確認する | 各エポック後 |

バッチサイズを大きくすると 1 回の処理は重くなります。ただし「大きいほど必ず安定する」とは
言えません。メモリが足りないときは小さくし、安定しているかどうかは **検証の指標** で確かめます。

#### The terms, all at once
| Term | Meaning | Value in this course |
|---|---|---|
| epoch | one full pass over the training set | 15 |
| batch size | images the data loader reads at once | 8 |
| gradient accumulation | how many batches are summed before one update | 8, from `nbs=64` and `batch=8` |
| warmup | a start-up period that ramps the learning rate and the accumulated batch count | 3 epochs |
| learning rate | size of each adjustment | starts at 0.01, then decays |
| loss | distance from the truth; lower is better | falls during training |
| train | the work that determines the numbers | — |
| val | checking progress during training | after each epoch |

A larger batch is heavier per step, but "larger is more stable" is not a rule. Reduce the batch when
you run out of memory, and judge stability on the **validation metrics**.


#### ファインチューニング
ゼロから学習するのではなく、**COCO で学習済みの `yolo26n.pt` から始めます**。
これを **ファインチューニング** と呼びます。

理由：COCO で学んだ「物の形を見る力」は、PVC の欠陥にも役立ちます。
少ない画像（64 枚）でも、形の見方を一から覚え直す必要がありません。

とはいえ、**64 枚は非常に少ない** です。必要になる枚数は、欠陥の種類・大きさ・ばらつき・
照明条件で変わります。「何枚あればよい」と一概には言えません。使えるかどうかは、
**代表的な実データ** で確かめます。

#### Fine-tuning
Instead of training from scratch we **start from `yolo26n.pt`, pretrained on COCO**.
This is **fine-tuning**.

Reason: the "shape understanding" learned on COCO transfers usefully to PVC defects, so even with
64 images the model does not have to relearn how to see.

Still, **64 images is very few**. How many you need depends on the defect types, their size, the
variation between parts and the lighting, so no single number applies. Whether the model is usable
is checked on **representative real data**.


---

## Part 2　学習の準備 / Preparing the run

#### `data.yaml` を今の場所に合わせる
`data.yaml` の `path` は、データセット生成時の絶対パスです。
フォルダを移動した場合は、**今の場所に書き直す** 必要があります。

下のセルは、`data.yaml` を **今の研修フォルダに合わせて書き直します**。
他の設定（クラス名など）はそのまま残します。

#### Point `data.yaml` at the current location
`path` in `data.yaml` is the absolute path from generation time. If you moved the
folder, it must be rewritten.

The next cell **rewrites `data.yaml` for the current course folder** while keeping every other
setting.

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
dataset_dir = ASSETS / "yolo_dataset"
data_yaml_path = dataset_dir / "data.yaml"

# 既存の設定を読み、path の行だけを今の場所に置き換える
# keep every line, but replace the "path:" line with the current location
lines = data_yaml_path.read_text(encoding="utf-8").splitlines()
updated = []
for line in lines:
    if line.startswith("path:"):
        updated.append(f"path: {dataset_dir.as_posix()}")
    else:
        updated.append(line)
data_yaml_path.write_text("\n".join(updated) + "\n", encoding="utf-8")

print(data_yaml_path.read_text(encoding="utf-8"))
print("data.yaml を更新しました / updated:", data_yaml_path)

#### 学習画像とラベルの対応を最終確認する
学習を始める前に、**画像とラベルが 1 対 1 で対応しているか** を確認します。
ここで失敗すると、学習が始まってから変な結果になります。

#### Final check that images and labels pair up
Before training, confirm that **every image has its label**. Failing here produces
strange results only after training has run.

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
report = []
for split in ("train", "val", "test"):
    images = {p.stem for p in (dataset_dir / "images" / split).glob("*.png")}
    labels = {p.stem for p in (dataset_dir / "labels" / split).glob("*.txt")}
    report.append({
        "split": split,
        "images": len(images),
        "labels": len(labels),
        "missing_labels": len(images - labels),
        "orphan_labels": len(labels - images),
    })

import pandas as pd
print(pd.DataFrame(report).to_string(index=False))

---

## Part 3　学習を実行する / Running the training

#### このセルの学習設定
CPU でも現実的な時間で終わる、**小さな設定** にしています。

| 設定 | 値 | 理由 |
|---|---|---|
| `epochs` | 15 | 少なすぎず、待ち時間も短い |
| `imgsz` | 320 | 元画像が 320 px。CPU でも扱える |
| `batch` | 8 | メモリと速度のバランス |
| `workers` | 0 | Windows のノートブックで安全 |
| `device` | `cpu` | GPU が無くても動く |
| `seed` | 2026 | 再現性のため |

`nbs`（勾配をためる単位、既定 64）と `warmup_epochs`（既定 3.0）は、このセルでは指定していません。
値は保存された `args.yaml` に記録されています。`batch=8` でも、更新は 8 バッチごとです。

> **待ち時間はハードウェアで大きく変わります。** 数分で終わることもあれば、
> 15 分以上かかることもあります。**終了時間は保証できません。**
> 遅い場合は `epochs` を 5 に下げてください（学習の流れは同じです）。

#### The configuration in this cell
The configuration is deliberately **small** so it finishes in a reasonable time on CPU.

| Setting | Value | Why |
|---|---|---|
| `epochs` | 15 | enough to see progress, short enough to wait |
| `imgsz` | 320 | the source images are 320 px; CPU friendly |
| `batch` | 8 | balance between memory and speed |
| `workers` | 0 | safe in Windows notebooks |
| `device` | `cpu` | works without a GPU |
| `seed` | 2026 | reproducibility |

`nbs` (the accumulation unit, default 64) and `warmup_epochs` (default 3.0) are not set in this
cell; the saved `args.yaml` records them. Even with `batch=8`, an update happens only every 8
batches.

**Runtime depends heavily on the hardware**: minutes on some machines, 15+ minutes on others.
**Completion time cannot be guaranteed.** If it is too slow, set `epochs = 5`; the learning process
is the same.


#### 実行スイッチ
次のセルを実行すると **学習が始まります**（CPU で数分かかります）。
スキップしたい場合は `RUN_TRAINING = False` に書き換えてください。

**長い学習（GPU 向け 100 エポック）は実行しません。** 例として文字列に保存してあります。

#### The run switch
Running the next cell **starts a short training job** (a few minutes on CPU).
Set `RUN_TRAINING = False` to skip it.

**The long GPU configuration (100 epochs) is never executed**; it is stored only as a string example.

### コードの読み方 / Reading the code

![](../../assets/svg/teach_epoch_batch_step.svg)

図：バッチ数と更新回数は同じとは限りません。（計算した合成例・模式図。モデルの実測出力ではありません。）

Figure: Batch count need not equal optimizer update count. (computed synthetic / schematic, not measured model output.)

**気づいてほしいこと**

バッチ数と更新回数は同じとは限りません。 入力と変換後の値を指でたどってください。

**What to notice**

Batch count need not equal optimizer update count. Trace the input and transformed values.

**考えてみてください / Try to answer**

64枚で batch=4 にすると、1 epoch のバッチ数はどうなりますか。

With 64 images and batch=4, how many batches make one epoch?


In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
# ================= 実行スイッチ / run switch =================
RUN_TRAINING = True         # 学習をスキップするときは False にしてください / set to False to skip
EPOCHS = 15                 # 遅い場合は 5 に下げてください / lower to 5 if it is slow
IMGSZ = 320
BATCH = 8
SEED = 2026
# =============================================================

RUN_LONG_GPU_EXAMPLE = """
# GPU がある場合の例（このノートブックでは実行しません）
# example for a machine with a GPU (not executed here)
model = YOLO("yolo26n.pt")
model.train(
    data=data_yaml_path.as_posix(),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,          # 0 = first GPU
    workers=4,
    seed=SEED,
    project=str(COURSE_ROOT / "training_results" / "day04"),
    name="pvc_defect_long",
)
"""

if not RUN_TRAINING:
    print("学習は実行していません。/ Training was not started.")
    print("理由 / reason: RUN_TRAINING = False")
    print()
    print("実行するには、このセルの RUN_TRAINING を True に書き換えて再実行してください。")
    print("Set RUN_TRAINING = True in this cell and run it again.")
    print("GPU 向けの長い設定の例は RUN_LONG_GPU_EXAMPLE に入っています（実行はしません）。")
    print()
    print("スキップした場合、以降のセルは動きません。第4日ノートブック03 も実行できません。")
    print("If you skip training, the following cells cannot run, nor can Day 4 Notebook 03.")
else:
    print("学習を開始します / starting training...")
    print(f"epochs={EPOCHS} imgsz={IMGSZ} batch={BATCH} seed={SEED}")

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
# 学習の実行 / run the training
if RUN_TRAINING:
    import time

    train_model = load_pretrained_yolo("yolo26n.pt")
    started = time.time()

    train_results = train_model.train(
        data=data_yaml_path.as_posix(),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        workers=0,                 # Windows のノートブックでは 0 が安全 / safe on Windows
        device="cpu",
        seed=SEED,
        project=str(COURSE_ROOT / "training_results" / "day04"),
        name="pvc_defect_quick",
        exist_ok=True,
        plots=True,
        verbose=False,
    )

    elapsed_minutes = (time.time() - started) / 60
    run_dir = Path(train_model.trainer.save_dir)
    from training_run import save_training_run
    training_manifest = save_training_run(COURSE_ROOT, run_dir, {
        "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "seed": SEED,
        "device": "cpu", "data": str(data_yaml_path.relative_to(COURSE_ROOT)),
    })
    print("評価へ渡す記録 / evaluation handoff:", training_manifest)
    print()
    print(f"学習にかかった時間 / training time: {elapsed_minutes:.1f} minutes")
    print("出力フォルダ / output folder:", run_dir)
else:
    run_dir = None
    print("RUN_TRAINING = False のためスキップしました / skipped because RUN_TRAINING is False")

#### 学習中の画面の読み方
`verbose=True` にすると、学習中に loss の行が出ます（この研修では静かにするため `False` にしています）。
次の列見出しは、この研修で使う ultralytics 8.4.211 の学習器が表示するものです。

```text
Epoch   GPU_mem  box_loss  cls_loss  l1_loss  Instances  Size
```

| 列 | 意味 |
|---|---|
| `Epoch` | 何周目か |
| `GPU_mem` | 使用メモリ（CPU 実行では 0 に近い値） |
| `box_loss` | 枠の位置の誤差。下がれば位置が合ってきた |
| `cls_loss` | クラス分類の誤差。下がれば種類を当てられるようになった |
| `l1_loss` | 枠の座標の誤差を L1（絶対値の平均）で測った値 |
| `Instances` | そのバッチに含まれたインスタンス（正解）の数 |
| `Size` | 入力画像の一辺の画素（この研修では 320） |

学習のあとに保存される `results.csv` には、同じ内容がエポックごとの列として残ります。
保存した実行（`pvc_defect_quick`）の列は次のとおりです。

```text
epoch,time,train/box_loss,train/cls_loss,train/l1_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/l1_loss,lr/pg0,lr/pg1,lr/pg2
```

`metrics/...` と `val/...` の列は、各エポックのあとに **検証データ** で計算した値です。
学習の損失だけを見ていると気づけない変化は、ここに出ます。

> 古い資料では `dfl_loss`（Distribution Focal Loss）を「境界の細かさの誤差」と説明しています。
> `dfl_loss` は DFL を使う **別のアーキテクチャ** の列で、この研修の YOLO26 には存在しません
> （`results.csv` にもありません）。読み替えず、`l1_loss` を使ってください。

損失が下がり続けていれば学習は進んでいます。ただし **損失だけでは良し悪しを決められません**。
`metrics/mAP50-95` などの検証指標とあわせて読みます。

#### How to read the training output
With `verbose=True` the loss lines appear during training (this course sets `False` to keep the
output quiet). These column headings come from the ultralytics 8.4.211 trainer used here:

```text
Epoch   GPU_mem  box_loss  cls_loss  l1_loss  Instances  Size
```

| Column | Meaning |
|---|---|
| `Epoch` | which pass |
| `GPU_mem` | memory used (close to 0 on CPU) |
| `box_loss` | box position error; falling means boxes fit better |
| `cls_loss` | class error; falling means classes are better |
| `l1_loss` | box-coordinate error measured as L1 (mean absolute error) |
| `Instances` | instances (ground-truth objects) in that batch |
| `Size` | input image side in pixels (320 here) |

`results.csv` keeps the same information as per-epoch columns after training. The recorded run
(`pvc_defect_quick`) has these columns:

```text
epoch,time,train/box_loss,train/cls_loss,train/l1_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/l1_loss,lr/pg0,lr/pg1,lr/pg2
```

The `metrics/...` and `val/...` columns are computed on the **validation** data after each epoch and
show changes that the training loss alone hides.

> Older material describes `dfl_loss` (Distribution Focal Loss) as "boundary refinement error".
> `dfl_loss` belongs to a **different architecture** that uses DFL and does not exist in this
> course's YOLO26 (it is absent from `results.csv`). Do not substitute it; use `l1_loss`.

Falling losses mean learning is progressing, but **the loss alone cannot judge the model**. Read it
together with validation metrics such as `metrics/mAP50-95`.


---

## Part 4　学習結果を保存する / Saving the results

### 学習 loss と検証 loss を分けて読む / read training and validation loss separately

![](../../assets/diagrams/teach_train_val_curves.png)

図：曲線の **形の例** です。あなたの学習結果ではありません。

Figure: an **example of the curve shape** — not your own training run.

**気づいてほしいこと**

図の **横軸は epoch（何周目か）**、**縦軸は loss（損失）** です。2 本の線は学習用と検証用で、
凡例で見分けます。学習 loss だけを見ると、epoch を増やすほど良くなったように見えます。
図の破線（「ここから過学習」）は形を示すための例示で、実際のデータでこの位置が決まるわけでは
ありません。**短い期間の横ばいや上昇だけで過学習と決められるわけでもありません。**

見るのは「検証側が下がり止まったか」「検証の指標（mAP50-95 など）が伸び続けているか」です。
実際の画像で見えた失敗の内容もあわせて使います。
**学習 loss が下がること自体は「良くなった」証拠になりません。**

**What to notice**

The **horizontal axis is the epoch** and the **vertical axis is the loss**. The two lines are the
training and validation curves; use the legend to tell them apart. Training loss alone makes more
epochs look better. The dashed marker ("overfitting starts here") is illustrative — no real dataset
fixes that point — and **a short plateau or a few rising epochs does not prove overfitting** either.

Look at whether the validation curve has stopped falling and whether the validation metrics (mAP50-95
and the others) are still improving, and add the failures you see on real images.
**A falling training loss is not evidence of a better model.**

**考えてみてください / Try to answer**

- 検証 loss が数 epoch 横ばいのとき、過学習だと言い切れますか。他に何を見ますか。
- If validation loss is flat for a few epochs, can you call it overfitting? What else would you
  check?


### 学習ログを 2 本の線で読む / reading the training log as two lines

![](../../assets/reference_outputs/ref_train_val_curves.png)

図：この研修で実際に行った短い学習（15 epoch, CPU）の記録。左は学習・検証の cls 損失、右は mAP50 と precision。保存された学習ログそのものです。

Figure: the short training run actually performed in this course (15 epochs, CPU). Left: train and val cls loss. Right: mAP50 and precision. This is the saved log.

**気づいてほしいこと**

読むのは絶対値より「両方が下がり続けるか」「差が開き続けないか」です。数 epoch の横ばいだけで過学習とは言えません。損失の列名は YOLO26 の実際の出力です（`train/l1_loss`・`val/l1_loss` があり、`dfl_loss` はありません）。

**What to notice**

Read whether both curves keep falling and whether the gap keeps widening, not a single value. A few flat epochs are not proof of overfitting. The loss columns are YOLO26's actual outputs (`train/l1_loss`, `val/l1_loss`; there is no `dfl_loss`).

**考えてみてください / Try to answer**

- 最後の 3 epoch で、mAP50 は上がっていますか、横ばいですか。
- In the last three epochs, is mAP50 rising or flat?


#### 4 つを保存する
学習が終わると、Ultralytics が出力フォルダに多くのファイルを作ります。
提出物としては、次の 4 つがあれば十分です。

| 種類 | ファイル | 用途 |
|---|---|---|
| 重み | `weights/best.pt` | 推論に使う。**最重要** |
| 設定 | `args.yaml` | どんな設定で学習したかの記録 |
| 指標 | `results.csv` | エポックごとの損失と精度 |
| ログ | `results.csv` から作る要約 | 提出用 |

`best.pt` は **検証データ（val）** で最も良かった時点の重み、`last.pt` は最後の epoch の重みです。
**ふつうは `best.pt` を使います。**

「最も良かった」の基準は、Ultralytics の **fitness** です。この研修で使う ultralytics 8.4.211 の
物体検出では、重みは `[0, 0, 0, 1]`（precision・recall・mAP50・mAP50-95 の順）です。
つまり **fitness = mAP50-95** で、`best.pt` は **検証の mAP50-95 が最も高かった epoch** の重みです。

```text
fitness = 1.0 × mAP50-95        （検証データで計算 / computed on val、ultralytics 8.4.211）
```

`best.pt` を選ぶときも **試験データは見ていません**。だから `best.pt` を選んだあとでも、
試験での評価は 1 回きりにできます。

`last.pt` は最後の epoch の重みです。学習の最終状態を残したいとき、`results.csv` を見て
別の epoch を自分で選びたいとき、`resume` で学習を続けたいときに使います。

#### Save four things
Ultralytics writes many files into the output folder. For submission you need four:

| Kind | File | Use |
|---|---|---|
| weights | `weights/best.pt` | for inference; **the important one** |
| config | `args.yaml` | the record of how training was configured |
| metrics | `results.csv` | loss and accuracy per epoch |
| log | a summary built from `results.csv` | for submission |

`best.pt` is the best checkpoint measured on the **validation** split; `last.pt` is the final
epoch. **Normally you use `best.pt`.**

Ultralytics **fitness** defines "best". In ultralytics 8.4.211 the detection weights are
`[0, 0, 0, 1]` over precision, recall, mAP50 and mAP50-95, so **fitness = mAP50-95**, and `best.pt`
is the epoch with the highest validation mAP50-95.

```text
fitness = 1.0 * mAP50-95        (computed on val, ultralytics 8.4.211)
```

Choosing `best.pt` never looks at the test split, so test can still be evaluated exactly once
afterwards.

`last.pt` is the final epoch. Keep it when you want the end state of the run, when you want to pick
a different epoch yourself from `results.csv`, or when you want to `resume` training.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
if run_dir is not None:
    best_weights = run_dir / "weights" / "best.pt"
    last_weights = run_dir / "weights" / "last.pt"
    args_file = run_dir / "args.yaml"
    results_csv = run_dir / "results.csv"

    for label, path in [("best.pt", best_weights), ("last.pt", last_weights),
                        ("args.yaml", args_file), ("results.csv", results_csv)]:
        exists = path.is_file()
        size = path.stat().st_size if exists else 0
        print(f"{label:<12} exists={exists}  size={size} bytes")

    # 学習結果を提出用フォルダにコピーする / copy the results for submission
    submission_dir = COURSE_ROOT / "submissions" / "day04"
    submission_dir.mkdir(parents=True, exist_ok=True)
    if best_weights.is_file():
        import shutil
        shutil.copyfile(best_weights, submission_dir / "best.pt")
        print()
        print("提出用にコピーしました / copied for submission:", submission_dir / "best.pt")
else:
    print("学習を実行していないため、保存するものはありません。")
    print("Nothing to save because training was not run.")

### 実行結果の例 / Expected output

この結果は、この研修で保存した実行の記録です（`yolo26n.pt`、15 epoch、`imgsz=320`、`batch=8`、seed 2026、CPU）。
細かな時間やファイルサイズは環境で変わりますが、4 つのファイルが作られたことを確認してください。

```text
学習を開始します / starting training...
epochs=15 imgsz=320 batch=8 seed=2026

学習にかかった時間 / training time: 2.3 minutes
出力フォルダ / output folder: .../training_results/day04/pvc_defect_quick

best.pt      exists=True  size=5416133 bytes
last.pt      exists=True  size=5416133 bytes
args.yaml    exists=True  size=2091 bytes
results.csv  exists=True  size=2062 bytes
```

These numbers come from the run recorded for this course (`yolo26n.pt`, 15 epochs, `imgsz=320`,
`batch=8`, seed 2026, CPU). Times and file sizes vary between machines; check that the four files
exist.

```text
starting training...
epochs=15 imgsz=320 batch=8 seed=2026

training time: 2.3 minutes
output folder: .../training_results/day04/pvc_defect_quick

best.pt      exists=True
last.pt      exists=True
args.yaml    exists=True
results.csv  exists=True
```


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
if run_dir is not None and results_csv.is_file():
    history = pd.read_csv(results_csv)
    history.columns = [c.strip() for c in history.columns]
    print("列 / columns:", list(history.columns))
    print()
    print(history.tail(5).to_string(index=False))

    # 提出用の要約 / a short summary for the submission
    summary_lines = [
        "# 学習ログの要約 / training log summary",
        "",
        f"epochs={EPOCHS} imgsz={IMGSZ} batch={BATCH} seed={SEED}",
        "",
        history.to_string(index=False),
    ]
    summary_path = COURSE_ROOT / "submissions" / "day04" / "training_log_summary.txt"
    summary_path.write_text("\n".join(summary_lines), encoding="utf-8")
    print()
    print("保存 / saved:", summary_path)
else:
    print("results.csv がありません / results.csv is not available")

#### 学習は成功したと言えるか
損失が下がっただけでは不十分です。**この段階では、まだ性能を確かめていません。**

次のノートブック（04_03）で、
**学習に使っていない画像** で性能を測ります。そこで初めて「使える／使えない」が分かります。

損失が横ばいになったとしても、それだけで過学習とは言えません。判断には
**検証の指標の推移**（`metrics/mAP50-95` など）と、実際の画像で見えた失敗の内容をあわせて
使います。また、実際に使えるかどうかは、**代表的な実データ** で確かめるまで分かりません。

この段階で「精度 95 %」のような数字を出すことは **しません**。
学習用データで測った数字は、実力を表さないからです。

#### Can we call this training successful?
Falling losses are not enough — **performance has not been verified at this stage**.

The next notebook (04_03) measures performance on **images never used for training**. Only then can
you say whether the model is usable.

A loss that stops falling is not by itself overfitting. Judge with the **trend of the validation
metrics** (`metrics/mAP50-95` and the others) and with the failures you see on real images. Whether
the system is usable stays unknown until it is checked on **representative real data**.

We deliberately do **not** quote an accuracy figure here: a number measured on training data does
not reflect real skill.


---

## 演習 / Exercises

#### 演習 1（基本）：学習を 5 エポックで実行する
`EPOCHS` を 5 にして学習を実行し、`box_loss` の最初と最後の値を記録してください。
あわせて `metrics/mAP50-95` の最初と最後も記録します。損失だけでは判断できないことを確かめます。

| エポック | box_loss | mAP50-95（val） |
|---|---|---|
| 1 | | |
| 5 | | |

- 完了の目安：両方の値を記録し、損失が下がったかどうかを説明できること
- 損失が下がらない、または横ばいでも、それだけで失敗とは言えません

#### Exercise 1 (basic): train for 5 epochs
Set `EPOCHS = 5`, run the training, and record the first and last `box_loss`. Record
`metrics/mAP50-95` for the same epochs — the loss alone is not enough to judge the run.

| Epoch | box_loss | mAP50-95 (val) |
|---|---|---|
| 1 | | |
| 5 | | |

- Done when both values are recorded and you can say whether the loss decreased
- A loss that stops falling, or stays flat, does not by itself mean the run failed


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 1 / Exercise 1
# EPOCHS = 5 にして、上の学習セルを再実行してください / set EPOCHS = 5 and re-run the training cell
# 記録 / record:
first_box_loss = None
last_box_loss = None
print("epoch 1 box_loss:", first_box_loss)
print("epoch 5 box_loss:", last_box_loss)

#### 演習 2（中級）：`imgsz` を変えて比べる
`IMGSZ` を 160 / 320 で学習し、**学習時間** と **同じ検証指標**（`metrics/mAP50-95`、または
precision・recall）を比べてください。学習 loss の最終値は、入力サイズが変わると数値の意味も
変わるため、精度の比較には使えません。
速さと精度のどちらを取るか、自分の言葉で書いてください。

- 完了の目安：2 回分の時間と検証指標が記録され、トレードオフが説明されていること

#### Exercise 2 (intermediate): compare `imgsz`
Train with `IMGSZ` 160 and 320, then compare **training time** with the **same validation metric**
(`metrics/mAP50-95`, or precision and recall). The final training loss is not an accuracy measure:
its meaning changes with the input size, so do not compare it across sizes. State which trade-off
you would choose and why.

- Done when both runs' times and validation metrics are recorded and the trade-off is explained


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 2 / Exercise 2
comparison = [
    {"imgsz": 160, "minutes": None, "val_map50_95": None},
    {"imgsz": 320, "minutes": None, "val_map50_95": None},
]
for row in comparison:
    print(row)


#### 演習 3（発展・任意）：学習が進まない原因を考える
損失が下がらない、または NaN になる場合の原因を 4 つ挙げ、
それぞれ「どう確認するか」を書いてください。**必須ではありません。**

完了の目安：原因 4 つと確認方法 4 つが書かれていること。

#### Exercise 3 (optional challenge): why training might not progress
List four reasons why the loss might not decrease (or becomes NaN), and for each one say how you
would check it. This exercise is **not required**.

Done when four causes and four checks are written.


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 3 / Exercise 3
causes = [
    ("", ""),
    ("", ""),
    ("", ""),
    ("", ""),
]
for index, (cause, check) in enumerate(causes, start=1):
    print(f"{index}. 原因 / cause: {cause or '-'} -> 確認 / check: {check or '-'}")

---

#### 理解度チェック / Understanding questions

1. エポック・バッチサイズ・重みの更新（optimizer step）は、それぞれ何が違いますか。
2. 損失（loss）が下がるとは、どういうことですか。それだけで性能を判断できますか。
3. ファインチューニングをする利点を 2 つ挙げてください。
4. `best.pt` は何を基準に選ばれますか。`last.pt` との違いは何ですか。
5. 学習が終わった直後に「精度は 95 % です」と言ってはいけないのはなぜですか。

1. How do an epoch, a batch size and an optimizer step differ?
2. What does it mean that the loss decreases, and is that enough to judge performance?
3. Give two benefits of fine-tuning.
4. What selects `best.pt`, and how does it differ from `last.pt`?
5. Why must you not announce "accuracy is 95%" right after training finishes?
Model answers: `docs/instructor/day04_instructor_guide.md`.


---

## 困ったときは / Troubleshooting

エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| 学習が始まらない | Training does not start | `RUN_TRAINING = False` になっています。学習するには `True` にしてください。 | `RUN_TRAINING` is still False. Set it to True and re-run. |
| `RuntimeError: Dataset ... not found` | `RuntimeError: Dataset ... not found` | `data.yaml` の `path` が古い可能性があります。Part 2 のセルで書き直してください。 | `path` in `data.yaml` may be stale. Re-run the Part 2 cell. |
| `CUDA out of memory` / `OutOfMemoryError` | `CUDA out of memory` / `OutOfMemoryError` | `BATCH` を 4、`IMGSZ` を 224 に下げてください。CPU 実行なら起きにくいです。 | Lower `BATCH` to 4 and `IMGSZ` to 224. Rare when running on CPU. |
| 学習が非常に遅い | Training is very slow | CPU 実行では数分〜十数分かかります。`EPOCHS` を 5、`IMGSZ` を 160 にしてください。 | CPU training takes minutes to tens of minutes. Use `EPOCHS=5`, `IMGSZ=160`. |
| `FileNotFoundError: yolo26n.pt` | `FileNotFoundError: yolo26n.pt` | モデルが未取得です。`assets/models/` にファイルがあるか確認してください。 | The model was never obtained. Check `assets/models/`. |
| 損失が下がらない | The loss does not decrease | ラベルの誤り、画像とラベルの不一致、学習率の問題などが考えられます。第4日01 の点検に戻ってください。 | Likely label errors, image/label mismatch, or the learning rate. Repeat the Day 4 Notebook 01 checks. |
| 学習ログに `dfl_loss` が無い | No `dfl_loss` column in the log | `dfl_loss` は DFL を使う別のアーキテクチャの列です。YOLO26 は `l1_loss` を使います。 | `dfl_loss` belongs to a different architecture that uses DFL. YOLO26 logs `l1_loss`. |
| 1 epoch の更新回数が合わない | The update count per epoch looks wrong | 1 batch = 1 回の更新とは限りません。`nbs` と `batch` による勾配の累積と warmup で決まります。 | One batch is not always one update: `nbs`, `batch` accumulation and warmup decide it. |


## 完了チェックリスト / Completion checklist

**必須の項目だけ** を並べています。**演習 3（発展・任意）は必須ではありません**。未実施でも完了です。
時間が余ったときだけ取り組んでください。

Only the **required** items are listed. **Exercise 3 (optional challenge) is not required**; leaving
it undone still counts as complete. Do it only if you have spare time.

すべてにチェックが付いたら、このノートブックは完了です。

This notebook is complete when every box is checked.

- [ ] Part 1〜Part 4 を読み、すべてのコードセルを実行した
      - [ ] Read Parts 1-4 and ran every code cell
- [ ] `data.yaml` を今の場所に合わせて書き直した
      - [ ] Rewrote `data.yaml` for the current location
- [ ] 画像とラベルの対応を確認した
      - [ ] Confirmed images and labels pair up
- [ ] 学習を実行した（`RUN_TRAINING = True`）
      - [ ] Ran the training (`RUN_TRAINING = True`)
- [ ] `best.pt` と `results.csv` が生成された
      - [ ] `best.pt` and `results.csv` exist
- [ ] 学習ログの要約を保存した
      - [ ] Saved the training summary
- [ ] 演習 1〜2（必須）を実施した
      - [ ] Completed exercises 1-2 (required)
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 questions
- [ ] ノートブックを保存した
      - [ ] Saved the notebook

### 任意の項目 / Optional items

- [ ] 演習 3（発展・任意）：学習が進まない原因を 4 つ挙げた
      - [ ] Exercise 3 (optional challenge): listed four possible causes


## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

- [Ultralytics Train モード](https://docs.ultralytics.com/modes/train/)
- [Ultralytics ハイパーパラメータ](https://docs.ultralytics.com/guides/hyperparameter-tuning/)
- [PyTorch 学習の基礎](https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

`BATCH` を 4 と 16 で学習し、損失の下がり方の「なめらかさ」を比べてください。

Train with `BATCH` 4 and 16 and compare how smoothly the loss decreases.